# House Prices: Advanced Regression Techniques
### Home Price Prediction (Kaggle Competition)
**Author:** Eduard Trubichkin  
**Date:** march 2026  
**Description:** Complete data analysis, preprocessing, modeling (sklearn, XGBoost), and linear regression implementation from scratch in numpy.

In [1]:
import pandas as pd

Creating 2 objects of the `pandas.DataFrame` class: train and test. These objects can be represented as a table in Excel, with columns and rows.

In [2]:
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

We use the method of the instance object `info()` (this method is available to all instances of the DataFrame class). We get information about the file, which states `RangeIndex: 1460 entries, 0 to 1459`, which means that there are only 1460 rows in the table. We also see `Data columns (total 81 columns)`, which means that there are 81 columns in the file. The names of the columns from 0 to 80 inclusive are listed below. `Non-Null Count` means the number of non-empty values. For example, `LostFrontage` has only 1201 missing positions out of 1460. `Dtype` means the data type.

In [3]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   str    
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   str    
 6   Alley          91 non-null     str    
 7   LotShape       1460 non-null   str    
 8   LandContour    1460 non-null   str    
 9   Utilities      1460 non-null   str    
 10  LotConfig      1460 non-null   str    
 11  LandSlope      1460 non-null   str    
 12  Neighborhood   1460 non-null   str    
 13  Condition1     1460 non-null   str    
 14  Condition2     1460 non-null   str    
 15  BldgType       1460 non-null   str    
 16  HouseStyle     1460 non-null   str    
 17  OverallQual    1460 non-null   int64  
 18  OverallCond    1460

Let's look at the basic statistics of SalePrice prices. This is done using the `describe()` method, which is a method of the `DataFrame` instance. This method is applicable both to the `DataFrame` object and to a specific column. Since we are specifically interested in price statistics, we will apply the method to a specific column.   
`pandas.DataFrame` is designed so that its columns can be accessed by name in two ways:   
1) `train['SalePrice']` - always works   
2) `train.SalePrice` - does not work if there are spaces or special characters in the name.   

The basic statistics include parameters such as: 
- count (the total number of parameters to make sure there are no gaps in the column)
- mean (average price)
- std (standard deviation, a measure of how much the values deviate from the average)
- min/max (extreme values)
- 25% (meaning that 25% of houses are cheaper, 75% more expensive)
- 50% (half of the houses are cheaper, half are more expensive)
- 75% (75% of the houses are cheaper, 25% are more expensive)

In [4]:
train['SalePrice'].describe()

count      1460.000000
mean     180921.195890
std       79442.502883
min       34900.000000
25%      129975.000000
50%      163000.000000
75%      214000.000000
max      755000.000000
Name: SalePrice, dtype: float64

The results should pay due attention to the variation from 75% to max. If the spread is large, this is a clear sign of outliers. That is, there are several extremely expensive houses in the data. The model may learn incorrectly from such outliers, so they are usually removed from the sample or transformations (logarithm) are used to smooth out the impact.  

Based on the data obtained, we can conclude that there is a huge gap between 75% and the maximum in the dataset (this needs to be fixed), and there are also many gaps in a number of parameters (this also needs to be fixed). Let's start working with the passes. Let's calculate their number (in absolute and percentage terms) and decide what to do with them (either deleting a column if there are too many gaps, or deleting a row if there are few gaps, or we can fill in something).

In [5]:
missing = train.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(missing)

PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
LotFrontage      259
GarageType        81
GarageYrBlt       81
GarageFinish      81
GarageQual        81
GarageCond        81
BsmtExposure      38
BsmtFinType2      38
BsmtQual          37
BsmtCond          37
BsmtFinType1      37
MasVnrArea         8
Electrical         1
dtype: int64


`isnull()` is a method of the `DataFrame` instance. It returns a new object of the same size, but consisting of Boolean values: `True` where there is no value, `False` where there is a value. The `sum()` applied to the `DataFrame` instance counts all `True` columns and outputs the sum: how many gaps are in each column. We end up with a `Series` object. The `Series` instance is just one component of the `DataFrames` instance, one column. Visually it looks like two columns, but in fact the first column is an index, which can be anything, even a number, even a row.   
After the table has been converted to a table containing values indicating the number of gaps, it is necessary to filter the column. To do this, the `missing=missing[missing>0]` construction is used, which leaves only those records where there is at least one pass. After that, the `sort_values()` method is applied (which applies to both `Series` and `DataFrame` instances), which sorts the remaining non-zero values. The argument `ascending=True` (used by default) is ascending sorting, `ascending=False` is descending sorting.   

After getting the absolute value of the number of passes, it is necessary to get the value in relative form. To do this, you can use the following algorithm: divide the instance of `Series` by the length of `train` and multiply by 100. `len(train)` returns the number of rows in the instance of `DataFrame'. Vectorized operations are available in pandas, that is, when dividing an instance of a Series, the division occurs piecemeal.

In [6]:
missing_percent = train.isnull().sum() / len(train) * 100
missing_percent = missing_percent[missing_percent > 0].sort_values(ascending=False)
print(missing_percent)

PoolQC          99.520548
MiscFeature     96.301370
Alley           93.767123
Fence           80.753425
MasVnrType      59.726027
FireplaceQu     47.260274
LotFrontage     17.739726
GarageType       5.547945
GarageYrBlt      5.547945
GarageFinish     5.547945
GarageQual       5.547945
GarageCond       5.547945
BsmtExposure     2.602740
BsmtFinType2     2.602740
BsmtQual         2.534247
BsmtCond         2.534247
BsmtFinType1     2.534247
MasVnrArea       0.547945
Electrical       0.068493
dtype: float64


Columns that have >90% omissions can be completely deleted, as they are almost completely empty and hardly carry any valuable information. However, before deleting it, you need to carefully look at the meaning of the attribute. Sometimes there are signs that have a null value, but not because they are empty, but because they indicate the absence/negation of some parameter. For example, the "Alley" parameter indicates the presence or absence of an "alley". And in this particular case, it would be more rational to replace the pass with "NoAlley".   

In general, in "train.csv", the "Alley" column is mostly set to "NA", which is not a skip. But when pandas reads CSV using pandas.read_csv(), it has a list of values that are considered skipped by default. This list includes, for example: 'NA', 'null', 'NaN', 'None', an empty string, and some others. That is why, when the `isnull()` method was applied, all these cells acquired the value `True`. Therefore, during data processing, it is necessary to take into account such moments and carefully monitor which columns really need to be removed, and in which columns the values need to be subdivided.   

Let's take a closer look at the "Alley" column. In order to understand the column structure, you can use the `value_counts()` method with the `dropna` argument. This is the `Series` instance method, it counts how many times each unique value occurs in a column. The result is also a `Series` object, where the indexes are unique values and the values are their frequencies. The `dropna` argument determines whether to include skips in the count. The default value is `True`. If it is necessary to include gaps in the calculation, then `dropna=False` is used.

In [7]:
print(train['Alley'].value_counts(dropna=False))

Alley
NaN     1369
Grvl      50
Pave      41
Name: count, dtype: int64


The results show that the "Alley" column, in addition to the "Nan" values, has values such as "Grvl" (gravel) and "Pave" (asphalt). Thus, it can be understood that the "NA" in the "Alley" column does not mean emptiness, but rather the absence of an alley near the house. This means that it is necessary to replace the values of "NA" with "NoAllay".   

To change the values, you can use the `fillna()` method, which can be applied to both `DataFrame` and `Series` objects. The purpose of this method is to fill in the missing values in the data. A string is used as an argument, which will be used to replace all the omissions.

In [8]:
train['Alley'] = train['Alley'].fillna('NoAlley')
print(train['Alley'].value_counts(dropna=False))

Alley
NoAlley    1369
Grvl         50
Pave         41
Name: count, dtype: int64


As you can see from the result, the string "NoAlley" is now used instead of an empty value.   

So, as mentioned earlier, columns with >90% omissions should be deleted if the omissions do not carry information. If the column has an average or small number of gaps, then they also need to be filled in, but using more "accurate" methods. For example, the median of the group, zero, mode, and so on, so as not to distort the distribution.   
In any case, there should be no gaps at all in the training sample, since most machine learning models cannot work with missing values. If there is at least one "NaN" left in the data, an error will be received when trying to train the model. Therefore, it is necessary to continue to identify gaps that carry information.  

There are a total of 19 columns in the CSV with at least one pass. The "Alley" column has already been disassembled, so we move on to the next columns.   
The file will help with the analysis "data_description.txt ", which contains a description of all the signs. If you open it and find, for example, "Alley" there, you can see that NA (or skip) means "No alley access", and not lack of data. Therefore, the analysis of the remaining columns will be based on a file with a description of the features.

- **PoolQC**: 99.520548% passes; "NA" means that there is no pool; Replace with "NoPool"
- **MiscFeature**: 96.301370% passes; "NA" means that there are no additional buildings; Replace with "NoMiscFeature"
- **Fence**: 80.753425% passes; "NA" means that there is no fence; Replace with "NoFence"
- **MasVnrType**: 59.726027% passes; "None" means that there is no cladding; Replace with "NoMasVnrType"
- **FireplaceQu**: 47.260274% passes; "NA" means that there is no fireplace; Replaced by "NoFireplaceQu"
- **LotFrontage**: 17.739726% of passes; "LotFrontage" displays the length of the street along the site; The passes can be filled in with the median for the area, since in one area the plots are usually similar. 
- **GarageType**: 5.547945% passes; "NA" means there is no garage; Заменим пропуски на "NoGarage".
- **GarageYrBlt**: 5.547945% passes; The year when the garage was built; a pass means that there is no garage; let's replace it with zero. 
- **GarageFinish**: 5.547945% passes; A sign indicating the finishing of the garage; "NA" means that there is no finishing; replace with "NoGarage".
- **GarageQual**: 5.547945% passes; The attribute indicates the quality of the garage; "NA" means that there is no garage; replace with "NoGarage". 
- **GarageCond**: 5.547945% passes; The sign indicates the condition of the garage; "NA" means that there is no garage; Replace with "NoGarage". 
- **BsmtExposure**: 2.602740% passes; In this feature, "NA" means that there is no basement; Let's replace it with "NoBasement". 
- **BsmtFinType2**: 2.602740% passes; In this feature, "NA" means that there is no basement; Let's replace it with "NoBasement".
- **BsmtQual**: 2.534247% of passes; In this feature, "NA" means that there is no basement; Let's replace it with "NoBasement".
- **BsmtCond**: 2.534247% passes; In this feature, "NA" means that there is no basement; Let's replace it with "NoBasement".
- **BsmtFinType1**: 2.534247% passes; In this feature, "NA" means that there is no basement; Let's replace it with "NoBasement".
- **MasVnrArea**: 0.547945% of passes; This parameter describes the area of masonry cladding in square feet. Using the string `print(train['MasVnrArea'].value_counts(dropna=False) / len(train) * 100)` you can find out what values are in the column at all and what percentage of the total they occupy. The value "0.0" has 58.972603% of the total number of all values. While the value of "NaN" is 0.547945%. Based on the meaning of the parameter, it would be most logical to fill in the gaps with zeros, since if we fill them in, for example, with the median, the model will think that the house has cladding, although it actually will not be. Fill in all the gaps with the value zero.
- **Electrical**: 0.068493% of passes; This parameter has only 1 pass; therefore, you can simply delete the row with this pass, as this does not affect the overall picture.

In [9]:
train['PoolQC'] = train['PoolQC'].fillna('NoPool')
print(train['PoolQC'].value_counts(dropna=False))

PoolQC
NoPool    1453
Gd           3
Ex           2
Fa           2
Name: count, dtype: int64


In [10]:
train['MiscFeature'] = train['MiscFeature'].fillna('NoMiscFeature')
print(train['MiscFeature'].value_counts(dropna=False))

MiscFeature
NoMiscFeature    1406
Shed               49
Gar2                2
Othr                2
TenC                1
Name: count, dtype: int64


In [11]:
train['Fence'] = train['Fence'].fillna('NoFence')
print(train['Fence'].value_counts(dropna=False))

Fence
NoFence    1179
MnPrv       157
GdPrv        59
GdWo         54
MnWw         11
Name: count, dtype: int64


In [12]:
train['MasVnrType'] = train['MasVnrType'].fillna('NoMasVnrType')
print(train['MasVnrType'].value_counts(dropna=False))

MasVnrType
NoMasVnrType    872
BrkFace         445
Stone           128
BrkCmn           15
Name: count, dtype: int64


In [13]:
train['FireplaceQu'] = train['FireplaceQu'].fillna('NoFireplaceQu')
print(train['FireplaceQu'].value_counts(dropna=False))

FireplaceQu
NoFireplaceQu    690
Gd               380
TA               313
Fa                33
Ex                24
Po                20
Name: count, dtype: int64


In the context of LotFrontage, all gaps will be filled in with the median for the area. In one area, the sections are usually similar in size, so the length of the frontal part should be close.   

The implementation consists of several stages:   
1) Calculating the median "LotFrontage" for each district  
2) Filling in gaps using `transform`  
3) Handling a special case in which all values of the "LotFrontage" column may be skipped in some area 
4) Verification 

The first and second steps can be combined:

In [15]:
train['LotFrontage'] = train.groupby('Neighborhood')['LotFrontage'].transform(
    lambda x: x.fillna(x.median())
)

To implement filling in gaps with the median, the `groupby()` method is used, which groups all DataFrame rows by unique values. In this case, the value "Neighborhood" is passed, and it is by the unique values of this column that the grouping will take place. The result of `groupby()` is a special `DataFrameGroupBy` object that stores information about groups. In fact, this is the same object, but sorted so that first there are lines with one "Neighborhood" value, then with another, then with a third, and so on.   

After grouping, you need to select the specific column you want to work with. In this case, it is the "LotFrontage" column. The entry `['LotFrontage']` means that you need to take only the value of the column "LotFrontage" from each group. After that, a `SeriesGroupBy` object is obtained - a set of series (one for each group), each of which contains the "LotFrontage" values for houses in a particular area. That is, the remaining 79 columns out of 81 are cut off. And the work is carried out with only 2 columns - "Neighborhood" and "LotFrontage". This results in a single `SeriesGroupBy` object that contains tables whose indexes are a unique ID value, and the values are "LotFrontage" values. But the groups are formed according to the unique meaning of the "Neighborhood" attribute.  

Next, the `transform()` method is used, which applies the function to each group, and then collects everything in a DataFrame. In `transform()`, `lambda` is passed, a small function without a name. It looks like: `lambda arguments: expression`. And in this case, the function has the form `lambda x: x.fillna(x.median())`, where `x` is the group `SeriesGroupBy`. And for each such group, the `fillna()` method is applied with the `median()` argument, which calculates the median. And so the gaps in each group are filled in with the median value of each group.
